# Regression: delivery-time forecast at two information points

**Business role.** At checkout, estimate purchase-to-delivery time with information then available. At carrier handover, update the estimate with elapsed time and recent route history. The report's operational use is the handover estimate, alongside the late-order classifier. Both stages use the same delivered-order cohort and chronological validation windows.

The executable training and preprocessing live in [`phase2_eta.py`](../../phase2_eta.py) and its imported helpers. This notebook reads the saved outputs from that run; it does not refit or reselect on the already examined later period. For a full rerun, execute `python phase2_eta.py` from `phase2_final/`, then rerun this notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "phase2_feature_spec.json").is_file()
             and (p / "phase2_eta.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from phase2_final/ or a folder beneath it")
RESULTS = ROOT / "results" / "phase2"
spec = json.loads((ROOT / "data" / "phase2_feature_spec.json").read_text())
print("Phase 2 folder:", ROOT)
print("Saved feature contract:", spec.get("version", "version unspecified"))

Phase 2 folder: /Users/joshualum/Documents/VSCode/IT5006_Grp_6/phase2_final
Saved feature contract: version unspecified


## Candidate comparison and validation selection

The script starts with mean and plain linear baselines, then compares ridge, tree, random forest and gradient boosting candidates. Mean validation MAE across five chronological windows determines the winner; among candidates within one standard error of the best, the simpler eligible candidate is selected. The full candidate definitions and fold logic are in the training script.

In [2]:
cv = pd.read_csv(RESULTS / "eta" / "cv_summary.csv")
selection = json.loads((RESULTS / "eta" / "selection.json").read_text())
for stage in ("checkout", "handover"):
    print(f"{stage}: {selection['reason'][stage]}")
    display(cv.loc[cv.stage.eq(stage), ["candidate", "algorithm", "mean_mae", "mean_rmse", "mean_r2"]]
            .sort_values("mean_mae").round(3).reset_index(drop=True))

checkout: forest_leaf20 has the lowest mean validation MAE (5.7829); ridge_100 (5.8085) is within one standard error and simpler, so it is chosen


,candidate,algorithm,mean_mae,mean_rmse,mean_r2
0,forest_leaf20,forest,5.783,9.340,0.176
1,ridge_100,ridge,5.809,9.398,0.167
2,boost_slow,boost,5.816,9.444,0.157
3,boost_baseline,boost,5.830,9.452,0.155
4,ridge_10,ridge,5.848,9.451,0.158
5,ridge_1,ridge,5.858,9.462,0.156
6,linear_baseline,linear,5.860,9.464,0.155
7,tree_depth6,tree,5.933,9.505,0.144
8,mean_baseline,mean,7.215,10.671,-0.078
9,tree_baseline,tree,7.787,12.344,-0.472


handover: ridge_100 has the lowest mean validation MAE (5.0985)


,candidate,algorithm,mean_mae,mean_rmse,mean_r2
0,ridge_100,ridge,5.099,8.406,0.338
1,voting_family_winners,voting,5.107,8.471,0.327
2,ridge_10,ridge,5.136,8.460,0.329
3,ridge_1,ridge,5.145,8.470,0.327
4,linear_baseline,linear,5.146,8.472,0.327
5,boost_baseline,boost,5.214,8.651,0.297
6,boost_slow,boost,5.215,8.666,0.294
7,elapsed_plus_route_baseline,rule,5.304,8.396,0.337
8,tree_depth6,tree,5.313,8.633,0.299
9,forest_leaf20,forest,5.346,8.808,0.270


## What the later period shows

These are descriptive later-period scores for the validation-selected ridge fit. The later period was inspected during development, so these results did not choose the model. Compare stages by MAE and inspect the monthly rows for changing conditions; do not interpret the point forecast as a calibrated customer promise.

In [3]:
test = pd.read_csv(RESULTS / "eta" / "test_metrics.csv")
display(test.loc[test.scope.eq("overall"), ["stage", "candidate", "orders", "mae", "rmse", "r2"]].round(3))
display(test.loc[~test.scope.eq("overall"), ["stage", "scope", "orders", "mae", "r2"]]
        .sort_values(["stage", "scope"]).round(3))

,stage,candidate,orders,mae,rmse,r2
0,checkout,ridge_100,19230,4.995,6.412,-0.175
5,checkout,boost_baseline,19230,4.708,6.216,-0.104
10,handover,ridge_100,19230,3.972,5.442,0.154
15,handover,boost_baseline,19230,3.706,5.249,0.212
20,handover,voting_family_winners,19230,3.786,5.255,0.211


,stage,scope,orders,mae,r2
1,checkout,2018-05,755,4.325,0.346
6,checkout,2018-05,755,4.341,0.322
2,checkout,2018-06,6050,5.552,-0.154
7,checkout,2018-06,6050,5.083,-0.044
3,checkout,2018-07,6119,5.151,-0.135
8,checkout,2018-07,6119,4.926,-0.089
4,checkout,2018-08,6306,4.390,-0.628
9,checkout,2018-08,6306,4.182,-0.558
11,handover,2018-05,755,4.153,0.391
16,handover,2018-05,755,3.845,0.457


## Rebuild the evidence

Run `python phase2_eta.py` from the `phase2_final/` folder to refit all candidates and regenerate `results/phase2/eta/`. The script keeps preprocessing inside scikit-learn pipelines and enforces the chronological feature and outcome availability rules. Then run this notebook to inspect the new saved outputs. The [report evidence script](../../reports/phase2_final_evidence.py) updates the report's generated tables and figures.